# Cleaning Pipeline 00 — Manifest, File Integrity, and Basic Metadata

This notebook explains the first cleaning stage: **inventory every training image before deciding what to remove**.

It answers which files exist, whether Pillow can open them, and what their dimensions, mode, aspect ratio, and file size are.

**Decision rule:** only clearly unreadable or corrupt files are eligible for automatic exclusion. Unusual dimensions or formats are review candidates, not automatic deletions.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "cleaning_pipeline":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src"))

DATA_ROOT = REPO_ROOT / "BDC2026"
OUTPUT_ROOT = REPO_ROOT / "eda_outputs" / "notebook_pipeline"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

LABEL2ID = {
    "0_Recyclable": 0,
    "1_Electronic": 1,
    "2_Organic": 2,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile, ImageOps
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_images(folder: Path):
    return sorted([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

rows = []
for class_name, label in LABEL2ID.items():
    class_dir = DATA_ROOT / "train" / class_name
    if not class_dir.exists():
        raise FileNotFoundError(class_dir)
    for path in list_images(class_dir):
        rows.append({
            "path": str(path.resolve()),
            "relative_path": str(path.relative_to(DATA_ROOT)),
            "class_name": class_name,
            "label": label,
        })

manifest = pd.DataFrame(rows)
print("Images found:", len(manifest))
display(manifest["class_name"].value_counts().sort_index().to_frame("count"))

In [ ]:
def inspect_image(path: str):
    try:
        file_path = Path(path)
        with Image.open(file_path) as img:
            img = ImageOps.exif_transpose(img)
            width, height = img.size
            mode = img.mode
            image_format = img.format
            img.verify()
        return {
            "is_valid": True,
            "width": width,
            "height": height,
            "mode": mode,
            "format": image_format,
            "file_size_bytes": file_path.stat().st_size,
            "error": "",
        }
    except Exception as exc:
        return {
            "is_valid": False,
            "width": np.nan,
            "height": np.nan,
            "mode": "",
            "format": "",
            "file_size_bytes": Path(path).stat().st_size if Path(path).exists() else np.nan,
            "error": repr(exc),
        }

metadata = pd.DataFrame([inspect_image(p) for p in tqdm(manifest["path"], desc="Checking images")])
manifest = pd.concat([manifest, metadata], axis=1)
manifest["aspect_ratio"] = manifest["width"] / manifest["height"]
manifest["min_side"] = manifest[["width", "height"]].min(axis=1)
display(manifest.head())

In [ ]:
summary = pd.Series({
    "num_images": len(manifest),
    "num_valid": int(manifest["is_valid"].sum()),
    "num_corrupt": int((~manifest["is_valid"]).sum()),
    "min_width": manifest["width"].min(),
    "max_width": manifest["width"].max(),
    "min_height": manifest["height"].min(),
    "max_height": manifest["height"].max(),
})
display(summary.to_frame("value"))

corrupt = manifest[~manifest["is_valid"]].copy()
display(corrupt[["relative_path", "class_name", "error"]].head(50))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
manifest["width"].dropna().hist(bins=50, ax=axes[0]); axes[0].set_title("Width distribution")
manifest["height"].dropna().hist(bins=50, ax=axes[1]); axes[1].set_title("Height distribution")
manifest["aspect_ratio"].dropna().hist(bins=50, ax=axes[2]); axes[2].set_title("Aspect-ratio distribution")
plt.tight_layout()
plt.show()

## Interpretation guide

- **Corrupt/unreadable:** safe to exclude after recording the path and error.
- **Very small images:** may lack detail, but can still be valid examples.
- **Extreme aspect ratios:** may be panoramas, crops, screenshots, or annotation mistakes. Review visually.
- **Unexpected modes:** convert during loading if possible; do not remove solely because the image is grayscale or RGBA.

In [ ]:
stage_dir = OUTPUT_ROOT / "00_manifest"
stage_dir.mkdir(parents=True, exist_ok=True)
manifest.to_csv(stage_dir / "train_manifest.csv", index=False)
corrupt.to_csv(stage_dir / "corrupted_images.csv", index=False)
print("Saved outputs to", stage_dir)